# Efficiency analysis

Choose a GPU runtime for research runs. Use a fixed commit and a unique Drive run name for every configuration. Reuse the run name and commit to resume. W&B defaults to offline in the Colab AGSM configuration.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
REPO_URL = os.environ.get("AGSM_REPO_URL") or input("GitHub clone URL: ").strip()
GIT_REF = (
    os.environ.get("AGSM_GIT_REF")
    or input("Commit, tag or branch [main]: ").strip()
    or "main"
)
REPO = Path("/content/agsm-2.0")


def shell(*args):
    subprocess.run(args, cwd=REPO, check=True)


if not REPO.exists():
    subprocess.run(["git", "clone", "--", REPO_URL, str(REPO)], check=True)
if (
    subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO, text=True
    ).strip()
    != REPO_URL
):
    raise RuntimeError("Existing checkout uses a different repository URL")
shell("git", "fetch", "origin", GIT_REF)
shell("git", "checkout", "--detach", "FETCH_HEAD")
shell("git", "rev-parse", "HEAD")
os.chdir(REPO)
shell(sys.executable, "-m", "pip", "install", "-e", ".[dev,tracking]")

In [ ]:
import re

import torch
import yaml

from agsm.config import load_config, merge, validate_config
from data.data_utils import make_dataset

CONFIG_PATH = "configs/training/colab.yaml"
VARIANT = None

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name())
CPU_SMOKE = not torch.cuda.is_available()
RUN_NAME = input("Drive run name [agsm-smoke]: ").strip() or "agsm-smoke"
if not re.fullmatch(r"[a-zA-Z0-9_-]+", RUN_NAME):
    raise ValueError("Use letters, numbers, underscores or hyphens in the run name")
OUTPUT = Path("/content/drive/MyDrive/agsm-runs") / RUN_NAME
CONFIG = load_config(CONFIG_PATH)
VARIANTS = CONFIG.pop("sweep", {})
if VARIANT:
    CONFIG = merge(CONFIG, VARIANTS[VARIANT])
if CPU_SMOKE:
    small = load_config("configs/training/smoke.yaml")
    CONFIG = merge(
        CONFIG,
        {key: small[key] for key in ("training", "data", "evaluation", "tracking")},
    )
    width_fields = (
        "model_dim",
        "key_dim",
        "value_dim",
        "slow_state_dim",
        "controller_dim",
        "refinement_hidden_dim",
    )
    if CONFIG["model"]["name"] in {"agsm_v2", "gru", "transformer"}:
        CONFIG["model"].update({key: small["model"][key] for key in width_fields})
    else:
        raise RuntimeError("This baseline requires a compatible CUDA runtime")
validate_config(CONFIG)
print(make_dataset(CONFIG, "train").metadata)
resolved = Path("/content/agsm_resolved.yaml")
resolved.write_text(yaml.safe_dump(CONFIG))

In [ ]:
command = [
    sys.executable,
    "-m",
    "experiments.run_experiment",
    "--config",
    str(resolved),
    "--output",
    str(OUTPUT),
]
if list(OUTPUT.glob("epoch-*.pt")):
    command.append("--resume")
shell(*command)
print((OUTPUT / "experiment_report.md").read_text())

In [ ]:
shell(
    sys.executable,
    "-m",
    "scripts.benchmark",
    "--checkpoint",
    str(OUTPUT / "best.pt"),
    "--output",
    str(OUTPUT / "profile.json"),
    "--lengths",
    *map(str, CONFIG["evaluation"]["sequence_lengths"]),
    "--profile",
)

In [ ]:
# Optional. Run only after selecting a public artifact and a pre-existing release.
# Set GH_TOKEN via Colab Secrets if gh authentication is needed.
# from google.colab import userdata
# os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
# shell('gh', 'release', 'upload', '<existing-tag>', str(OUTPUT / 'best.pt'),
#       '--repo', '<owner>/<repo>')
